In [1]:
import os
import shutil
import json
import re

In [8]:
def get_models(root_folder):
    valid_subfolders = []
    for subfolder_name in os.listdir(root_folder):
        subfolder_path = os.path.join(root_folder, subfolder_name)
        if os.path.isdir(subfolder_path):
            output_folder_path = os.path.join(subfolder_path, 'output')
            if os.path.isdir(output_folder_path) and os.listdir(output_folder_path):
                valid_subfolders.append(subfolder_name)
    return valid_subfolders

In [10]:
models = get_models('model_output')
model_name = models[1]
models

['Llama-3-8B-Instruct-Gradient-1048k_5_shot',
 'OpenBioLLM-Llama3-70B_3_shot',
 'Llama-3-70B-Instruct_3_shot']

In [11]:
label_folder = "../../input/lct_p3"
model_folder = f"model_output/{model_name}/output"
ready_folder = f"model_output/{model_name}/ready"
failure_folder = f"model_output/{model_name}/failure"

categories = ['Condition', 'Observation', 'Drug']



#### Filter failure model files

In [12]:
def read_and_process_files(model_name):
    model_folder = f"model_output/{model_name}/output"
    ready_folder = f"model_output/{model_name}/ready"
    failure_folder = f"model_output/{model_name}/failure"
    os.makedirs(ready_folder, exist_ok=True)
    os.makedirs(failure_folder, exist_ok=True)

    def visit(node, category, entities):
        if isinstance(node, dict):
            if category in node:
                entities.extend(node[category])
            for value in node.values():
                visit(value, category, entities)
        elif isinstance(node, list):
            for item in node:
                visit(item, category, entities)

    for filename in os.listdir(model_folder):
        if filename.endswith(".json"):
            file_path = os.path.join(model_folder, filename)
            try:
                with open(file_path, 'r', encoding="utf-8") as file:
                    data = json.load(file)
                entities = []
                visit(data, 'category', entities)
                shutil.copy(file_path, ready_folder)
            except Exception as e:
                shutil.copy(file_path, failure_folder)
                print(f"Error processing file {filename}: {e}")

read_and_process_files(model_name)

Error processing file OpenBioLLM-Llama3-70B_NCT03860025_exc_3_shot.json: Expecting value: line 1 column 1 (char 0)
Error processing file OpenBioLLM-Llama3-70B_NCT03860337_inc_3_shot.json: Expecting value: line 1 column 1 (char 0)
Error processing file OpenBioLLM-Llama3-70B_NCT03861130_inc_3_shot.json: Extra data: line 17 column 1 (char 515)
Error processing file OpenBioLLM-Llama3-70B_NCT03862404_exc_3_shot.json: Extra data: line 58 column 3 (char 1268)
Error processing file OpenBioLLM-Llama3-70B_NCT03863561_exc_3_shot.json: Expecting value: line 1 column 1 (char 0)
Error processing file OpenBioLLM-Llama3-70B_NCT03864705_exc_3_shot.json: Expecting value: line 1 column 1 (char 0)
Error processing file OpenBioLLM-Llama3-70B_NCT03865004_exc_3_shot.json: Expecting value: line 1 column 1 (char 0)
Error processing file OpenBioLLM-Llama3-70B_NCT03865550_exc_3_shot.json: Extra data: line 72 column 3 (char 2102)
Error processing file OpenBioLLM-Llama3-70B_NCT03865589_inc_3_shot.json: Extra data:

In [13]:
def visit(node, category, entities):
    if isinstance(node, dict):
        if category in node:
            entities.extend(node[category])
        for value in node.values():
            visit(value, category, entities)
    elif isinstance(node, list):
        for item in node:
            visit(item, category, entities)

def calculate_metrics(label_data, model_data, metrics):
    for category in categories:
        label_entities = []
        visit(label_data, category, label_entities)
        model_entities = []
        visit(model_data, category, model_entities)

        true_positives = sum(entity in label_entities for entity in model_entities)
        false_positives = sum(entity not in label_entities for entity in model_entities)
        false_negatives = sum(entity not in model_entities for entity in label_entities)

        metrics[category]['true_positives'] += true_positives
        metrics[category]['false_positives'] += false_positives
        metrics[category]['false_negatives'] += false_negatives

def process_files(label_folder, model_folder):
    metrics = {category: {'true_positives': 0, 'false_positives': 0, 'false_negatives': 0} for category in categories}
    for filename in os.listdir(model_folder):
        if filename.endswith('.json'):
            model_path = os.path.join(model_folder, filename)
            match = re.search(r'NCT\d+_(?:exc|inc)', filename)
            if match:
                label_filename = f"{match.group(0)}_p3.json"
                label_path = os.path.join(label_folder, label_filename)

                if os.path.exists(label_path):
                    try:
                        with open(label_path, 'r', encoding='utf-8') as label_file, open(model_path, 'r', encoding='utf-8') as model_file:
                            label_data = json.load(label_file)
                            model_data = json.load(model_file)
                            calculate_metrics(label_data, model_data, metrics)
                    except json.JSONDecodeError as e:
                        print(f"Error decoding JSON file: {model_path}")
                        print(f"Error message: {str(e)}")


    for category in categories:
        true_positives = metrics[category]['true_positives']
        false_positives = metrics[category]['false_positives']
        false_negatives = metrics[category]['false_negatives']

        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = true_positives / (true_positives + false_positives + false_negatives) if (true_positives + false_positives + false_negatives) > 0 else 0

        metrics[category]['precision'] = precision
        metrics[category]['recall'] = recall
        metrics[category]['f1'] = f1
        metrics[category]['accuracy'] = accuracy

    return metrics

In [42]:
metrics = process_files(label_folder, ready_folder)

print("Metrics:")
for category, scores in metrics.items():
    print(f"{category}:")
    print(f"  Precision: {scores['precision']:.4f}")
    print(f"  Recall: {scores['recall']:.4f}")
    print(f"  F1-score: {scores['f1']:.4f}")
    print(f"  Accuracy: {scores['accuracy']:.4f}")

Metrics:
Condition:
  Precision: 0.4435
  Recall: 0.5511
  F1-score: 0.4915
  Accuracy: 0.3258
Observation:
  Precision: 0.2839
  Recall: 0.5012
  F1-score: 0.3625
  Accuracy: 0.2214
Drug:
  Precision: 0.7568
  Recall: 0.5600
  F1-score: 0.6437
  Accuracy: 0.4746


### Alte Funktion

In [ ]:
import os
import shutil
import json
import re

model_name = "Llama-3-70B-Instruct_3_shot"

label_folder = "../../input/lct_p3"
model_folder = f"model_output/{model_name}/output"
ready_folder = f"model_output/{model_name}/ready"
failure_folder = f"model_output/{model_name}/failure"

categories = ['Condition']

def visit(node, category, entities):
    if isinstance(node, dict):
        if category in node:
            entities.extend(node[category])
        for value in node.values():
            visit(value, category, entities)
    elif isinstance(node, list):
        for item in node:
            visit(item, category, entities)

def calculate_metrics(label_data, model_data, metrics):
    for category in categories:
        label_entities = []
        visit(label_data, category, label_entities)
        model_entities = []
        visit(model_data, category, model_entities)

        true_positives = sum(entity in label_entities for entity in model_entities)
        false_positives = sum(entity not in label_entities for entity in model_entities)
        false_negatives = sum(entity not in model_entities for entity in label_entities)

        metrics[category]['true_positives'] += true_positives
        metrics[category]['false_positives'] += false_positives
        metrics[category]['false_negatives'] += false_negatives

def process_files(label_folder, model_folder, ready_folder, failure_folder):
    num_files = 0
    num_ready_files = 0
    num_failure_files = 0

    os.makedirs(ready_folder, exist_ok=True)
    os.makedirs(failure_folder, exist_ok=True)

    for filename in os.listdir(model_folder):
        if filename.endswith('.json'):
            num_files += 1
            model_path = os.path.join(model_folder, filename)
            match = re.search(r'NCT\d+_(?:exc|inc)', filename)
            print(match)
            if match:
                label_filename = f"{match.group(0)}_p3.json"
                print(label_filename)
                label_path = os.path.join(label_folder, label_filename)

                if os.path.exists(label_path):
                    try:
                        with open(model_path, 'r', encoding='utf-8') as model_file:
                            json.load(model_file)
                        shutil.copy(model_path, os.path.join(ready_folder, filename))
                        num_ready_files += 1
                    except json.JSONDecodeError as e:
                        print(f"Error decoding JSON file: {model_path}")
                        print(f"Error message: {str(e)}")
                        shutil.copy(model_path, os.path.join(failure_folder, filename))
                        num_failure_files += 1

    print(f"Total files in the output folder: {num_files}")
    print(f"Files processed successfully: {num_ready_files}")
    print(f"Files with errors: {num_failure_files}")

    return num_ready_files

def calculate_metrics_from_ready_folder(label_folder, ready_folder):
    metrics = {category: {'true_positives': 0, 'false_positives': 0, 'false_negatives': 0} for category in categories}
    num_ready_files = 0

    for filename in os.listdir(ready_folder):
        if filename.endswith('.json'):
            model_path = os.path.join(ready_folder, filename)
            match = re.search(r'NCT\d+_(?:exc|inc)', filename)
            if match:
                label_filename = f"{match.group(0)}_p3.json"
                label_path = os.path.join(label_folder, label_filename)

                if os.path.exists(label_path):
                    with open(label_path, 'r', encoding='utf-8') as label_file, open(model_path, 'r', encoding='utf-8') as model_file:
                        label_data = json.load(label_file)
                        model_data = json.load(model_file)
                        calculate_metrics(label_data, model_data, metrics)
                    num_ready_files += 1

    if num_ready_files > 0:
        for category in categories:
            true_positives = metrics[category]['true_positives']
            false_positives = metrics[category]['false_positives']
            false_negatives = metrics[category]['false_negatives']

            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
            recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            accuracy = true_positives / (true_positives + false_positives + false_negatives) if (true_positives + false_positives + false_negatives) > 0 else 0

            metrics[category]['precision'] = precision
            metrics[category]['recall'] = recall
            metrics[category]['f1'] = f1
            metrics[category]['accuracy'] = accuracy
    else:
        print("No matching files found in the ready folder.")

    return metrics

num_ready_files = process_files(label_folder, model_folder, ready_folder, failure_folder)

if num_ready_files > 0:
    metrics = calculate_metrics_from_ready_folder(label_folder, ready_folder)

    print("Metrics:")
    for category, scores in metrics.items():
        print(f"{category}:")
        print(f"  Precision: {scores['precision']:.4f}")
        print(f"  Recall: {scores['recall']:.4f}")
        print(f"  F1-score: {scores['f1']:.4f}")
        print(f"  Accuracy: {scores['accuracy']:.4f}")
else:
    print("No files to calculate metrics from.")